# Kaggle K-Fold Execution
This notebook is designed to run on Kaggle Kernels. It assumes you have uploaded `colab_all_kfold.zip` as a Kaggle Dataset.

# Setup & Data Extraction
We need to extract the dataset to the working directory because Kaggle input datasets are read-only.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

# Paths
KAGGLE_INPUT_DIR = Path('/kaggle/input')
# Look for the uploaded zip - usually under the dataset name you gave it
zips = list(KAGGLE_INPUT_DIR.glob('**/*.zip'))
if not zips:
    raise FileNotFoundError("Could not find colab_all_kfold.zip in input directories!")
ZIP_PATH = zips[0]
print(f"Found zip: {ZIP_PATH}")

WORK_DIR = Path('/kaggle/working/PROJECT')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting zip... (this takes ~1-2 mins)")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/kaggle/working/')
print("Extraction complete.")

# Environment Setup
Install dependencies if needed (Kaggle usually has torch/pandas etc).

In [ ]:
os.chdir(WORK_DIR)
print(f"Current working directory: {os.getcwd()}")

# Fix Windows Path issue in CSVs (same as Colab)
# Kaggle uses Linux, so we need forward slashes and relative paths
ROOT = WORK_DIR
SPLITS = ROOT / 'product/artifacts/splits'
WIN_PREFIX = r'C:\FYP\PROJECT'

fixed = 0
for csv_file in SPLITS.glob('*.csv'):
    text = csv_file.read_text()
    if WIN_PREFIX in text:
        text = text.replace(WIN_PREFIX, str(ROOT) + '/')
        text = text.replace('\\', '/')
        csv_file.write_text(text)
        fixed += 1
print(f"Fixed paths in {fixed} CSV files.")

# Execution Function
Runs the training script and saves results.

In [ ]:
import subprocess
import sys
import json

def run_kfold(dataset, models):
    print(f"\n[{dataset.upper()}] Starting K-Fold Runs...")
    RESULTS_DIR = ROOT / 'product/artifacts/runs' / dataset
    
    for model in models:
        for fold in range(5):
            # SKIP LOGIC
            if dataset == 'pitt': continue
            if dataset == 'esc50':
                if model == 'mobilenetv2': continue
                if model == 'resnet50' and fold <= 3: continue 
            
            run_id = f"{dataset}_{model}_fold{fold}"
            summary_path = RESULTS_DIR / run_id / 'summary.json'
            
            if summary_path.exists():
                print(f"SKIP {run_id} (Already Done)")
                continue
                
            print(f">>> RUNNING {run_id}")
            
            # Params
            args = []
            if dataset == 'pitt':
                args = ['--epochs', '30', '--batch_size', '32', '--weighted_loss', '--lr', '1e-5', '--dropout', '0.7', '--weight_decay', '0.1', '--unfreeze_at', '10']
            else:
                args = ['--epochs', '30', '--batch_size', '32']
                
            cmd = [sys.executable, '-u', 'product/training/train_unified.py', 
                   '--dataset', dataset, 
                   '--model_type', model, 
                   '--fold', str(fold)] + args
                   
            # Run and stream output
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            for line in process.stdout:
                print(line, end='')  # Stream to cell output
            
            process.wait()
            if process.returncode != 0:
                print(f"!!! ERROR in {run_id} !!!")
            else:
                print(f"--- Finished {run_id} ---")

# Run the Experiments
**ESC-50**: ResNet50 (Fold 4), Hybrid (All)
**EmoDB**: All Models, All Folds

In [ ]:
# ESC-50 Remaining
run_kfold('esc50', ['resnet50', 'mobilenetv2', 'hybrid'])

# EmoDB Remaining
run_kfold('emodb', ['resnet50', 'mobilenetv2', 'hybrid'])

# Zip Results for Download
After running, zip the results folder so you can download it easily from the 'Output' tab.

In [ ]:
output_zip = '/kaggle/working/kfold_kaggle_results.zip'
print(f"Zipping results to {output_zip}...")

def zip_dir(path, ziph):
    for root, dirs, files in os.walk(path):
        for file in files:
            ziph.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), os.path.join(path, '..')))

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zip_dir(ROOT / 'product/artifacts/runs', zipf)

print("Done! Download kfold_kaggle_results.zip from the Output tab.")